In [1]:
import os
import pandas as pd

def update_agent_manager_dim(base_path):
    # --- Define Paths ---
    silver_path = os.path.join(base_path, "Silver")
    gold_path = os.path.join(base_path, "Gold")

    # --- Find Newest Dim in Silver ---
    silver_dims = [
        f for f in os.listdir(silver_path) 
        if f.startswith("dim_clean") and f.endswith(".csv")
    ]

    if not silver_dims:
        print("No dim_clean files found in Silver.")
        return

    # Identify the absolute newest version
    latest_silver_file = max(
        silver_dims, 
        key=lambda x: os.path.getmtime(os.path.join(silver_path, x))
    )

    source_full_path = os.path.join(silver_path, latest_silver_file)
    target_gold_file = os.path.join(gold_path, "agent_manager_dim.csv")

    # --- Clean up Gold Folder (Delete if any) ---
    if os.path.exists(target_gold_file):
        os.remove(target_gold_file)
        print("Deleted old agent_manager_dim.csv")

    # --- Read and Save as CSV ---
    df = pd.read_csv(source_full_path)
    
    # Save directly to gold as the specific filename
    df.to_csv(target_gold_file, index=False)
    
    print(f"Success: {latest_silver_file} saved as agent_manager_dim.csv in Gold.")



In [2]:
# ==========================================
# How to call the procedure:
# ==========================================
my_base_path = r"C:\CallAgents"
update_agent_manager_dim(my_base_path)

Deleted old agent_manager_dim.csv
Success: dim_clean_20200301.csv saved as agent_manager_dim.csv in Gold.


In [3]:
import os
import sys
import pandas as pd
from pandas.errors import EmptyDataError

def update_gold_schedules(base_path):
    # --- 1. PATHS ---
    silver_path = os.path.join(base_path, "Silver")
    gold_path = os.path.join(base_path, "Gold")

    # --- 2. READ ALL SCHEDULE FILES FROM SILVER ---
    sch_files = [
        f for f in os.listdir(silver_path)
        if f.startswith("Schedules") and f.endswith(".csv")
    ]

    if not sch_files:
        print("No Schedule files found in Silver.")
        return

    # Load all files into a single DataFrame
    dfs = [
        pd.read_csv(os.path.join(silver_path, f))
        for f in sch_files
    ]
    result_df = pd.concat(dfs, ignore_index=True)

    # --- 3. READ GOLD FILE SAFELY ---
    gold_file = os.path.join(gold_path, "Schedules.csv")

    if os.path.exists(gold_file) and os.path.getsize(gold_file) > 0:
        try:
            gold_df = pd.read_csv(gold_file)
        except EmptyDataError:
            gold_df = pd.DataFrame(columns=result_df.columns)
    else:
        gold_df = pd.DataFrame(columns=result_df.columns)

    # --- 4. CONCATENATE WITH GOLD ---
    if not result_df.empty:
        if gold_df.empty:
            final_df = result_df.copy()
        else:
            final_df = pd.concat([gold_df, result_df], ignore_index=True)
    else:
        final_df = gold_df

    # --- 5. SAVE TO GOLD ---
    final_df.to_csv(gold_file, index=False)

    print(f"Success: Schedules.csv updated in {gold_path}")
    print(f"Total rows in Gold: {len(final_df)}")




In [4]:
# ==========================================
# Example Call:
# ==========================================
my_base_path = r"C:\CallAgents"
update_gold_schedules(my_base_path)

Success: Schedules.csv updated in C:\CallAgents\Gold
Total rows in Gold: 27755


In [1]:
import os
import sys
import pandas as pd
from pandasql import sqldf
from pandas.errors import EmptyDataError

def update_gold_telephony(base_path):
    # 1. PATHS
    silver_path = os.path.join(base_path, "Silver")
    gold_path = os.path.join(base_path, "Gold")

    # 2. READ SILVER FILES
    tele_files = [
        f for f in os.listdir(silver_path)
        if f.startswith("Telephony") and f.endswith(".csv")
    ]

    if not tele_files:
        print("No Telephony files found in Silver.")
        return

    dfs = [pd.read_csv(os.path.join(silver_path, f)) for f in tele_files]
    silver_df = pd.concat(dfs, ignore_index=True)

    # 3. FILTER AND DEDUPLICATE (SQL)
    # Use locals() so sqldf can see the silver_df variable inside this function
    query = """
        SELECT *
        FROM (
            SELECT *,
                   ROW_NUMBER() OVER (
                       PARTITION BY employee_id, date
                       ORDER BY load_timestamp DESC
                   ) as rn
            FROM silver_df
            WHERE flag_dateWrong = 0 
              AND flag_nameWrong = 0 
        )
        WHERE rn = 1
    """
    # FIX: Pass locals() directly here
    result_df = sqldf(query, locals())

    # 4. DROP FLAG COLUMNS
    flag_cols = [c for c in result_df.columns if c.startswith("flag")]
    result_df = result_df.drop(columns=flag_cols + ["rn"])

    # 5. READ GOLD FILE SAFELY
    gold_file = os.path.join(gold_path, "Telephony.csv")

    if os.path.exists(gold_file) and os.path.getsize(gold_file) > 0:
        try:
            gold_df = pd.read_csv(gold_file)
        except EmptyDataError:
            gold_df = pd.DataFrame(columns=result_df.columns)
    else:
        gold_df = pd.DataFrame(columns=result_df.columns)

    # 6. CONCATENATE WITH GOLD
    if not result_df.empty:
        if gold_df.empty:
            final_df = result_df.copy()
        else:
            final_df = pd.concat([gold_df, result_df], ignore_index=True)
    else:
        final_df = gold_df

    # 7. SAVE TO GOLD
    final_df.to_csv(gold_file, index=False)
    print(f"Success: Telephony.csv updated in {gold_path}")
    print(f"Total rows in Gold: {len(final_df)}")




In [12]:
# ==========================================
# Example Call:
# ==========================================
my_base_path = r"C:\CallAgents"
update_gold_telephony(my_base_path)

Success: Telephony.csv updated in C:\CallAgents\Gold
Total rows in Gold: 666
